In [1]:
#======================================================================
# SECTION: MATRIX RANK ARITHMETIC (SUM & PRODUCT)
# COMPLEXITY: O(m * n * k) for multiplication, O(m * n) for addition.
# DATA SHAPE: Left (7 x 4) @ Right (4 x 7) -> Product (7 x 7).
# MATH: rank(A + B) <= rank(A) + rank(B) | rank(A @ B) <= min(rank(A), rank(B))
# TOOLS: np.random.rand(), @ (matmul), np.linalg.matrix_rank()
# PROTOCOL: Think-First Whiteboard Applied
#======================================================================

import numpy as np
import random

# set global print options
np.set_printoptions(suppress=True, precision=3)

## 1. Rank of Summed and Multiplied Matrices
Based on [Lecture 64](https://wgu.udemy.com/course/linear-algebra-theory-and-implementation/learn/lecture/10500648#overview) and [Lecture 68](https://wgu.udemy.com/course/linear-algebra-theory-and-implementation/learn/lecture/10500648#overview), these rules define the maximum possible information (dimensions) preserved through operations.

* **Multiplication Rule (The Bottleneck):**
    $$rank(AB) \leq \min(rank(A), rank(B))$$
    * *Logic:* You cannot create more information than what is available in the "narrowest" part of the data pipeline.

* **Addition Rule:**
    $$rank(A + B) \leq rank(A) + rank(B)$$
    * *Logic:* Summing two matrices can combine their spans, but the total rank can never exceed the sum of the individual ranks.

In [2]:
LEFT_MATRIX_SHAPE = (7, 4)
RIGHT_MATRIX_SHAPE = (4, 7)


def create_rank_deficient_matrix(shape):
    """Create a random matrix with at least one dependent row."""
    rows, cols = shape
    if rows < 2:
        raise ValueError("Matrix must have at least two rows to create a dependent row.")

    # Start with a random matrix of the requested shape.
    matrix = np.random.rand(*shape)

    # Make one row linearly dependent on another random row.
    row_indices = list(range(rows))
    row_to_copy_from, row_to_overwrite = random.sample(row_indices, 2)
    matrix[row_to_overwrite, :] = matrix[row_to_copy_from, :]

    return matrix


def print_matrix_with_rank(label, matrix):
    """Print a matrix and its numerical rank."""
    # NumPy computes numerical rank using singular values.
    matrix_rank = np.linalg.matrix_rank(matrix)

    print(f"{label}:")
    print(matrix)
    print(f"Rank of {label.lower()}: {matrix_rank}\n")


# Create a 7x4 matrix.
# A random rectangular matrix usually has full possible rank: min(7, 4) = 4.
left_matrix = np.random.rand(*LEFT_MATRIX_SHAPE)

# Create a 4x7 matrix with a repeated row.
# Because one row is duplicated, its rank should be less than the maximum possible rank of 4.
right_matrix = create_rank_deficient_matrix(RIGHT_MATRIX_SHAPE)

# Add matrices with matching dimensions.
# right_matrix is 4x7, so transposing it gives 7x4, matching left_matrix.
summed_matrix = left_matrix + right_matrix.T

# Matrix multiplication is valid because the inner dimensions match:
# left_matrix is 7x4 and right_matrix is 4x7, so the product is 7x7.
product_matrix = left_matrix @ right_matrix

print("Original matrices:\n")
print_matrix_with_rank("Left matrix", left_matrix)
print_matrix_with_rank("Right matrix", right_matrix)

# The rank of a sum is bounded by the sum of the individual ranks,
# but it also cannot exceed the matrix's maximum possible rank.
print_matrix_with_rank("Summed matrix", summed_matrix)

# The rank of a product is bounded by the lower-rank factor.
# Since right_matrix is intentionally rank-deficient, the product rank should reflect that bottleneck.
print_matrix_with_rank("Product matrix", product_matrix)

Original matrices:

Left matrix:
[[0.473 0.107 0.675 0.06 ]
 [0.646 0.424 0.3   0.696]
 [0.657 0.364 0.631 0.666]
 [0.299 0.974 0.253 0.081]
 [0.585 0.9   0.959 0.338]
 [0.389 0.401 0.151 0.547]
 [0.921 0.298 0.419 0.805]]
Rank of left matrix: 4

Right matrix:
[[0.145 0.594 0.148 0.873 0.02  0.966 0.689]
 [0.061 0.65  0.77  0.947 0.648 0.179 0.833]
 [0.551 0.45  0.043 0.085 0.049 0.629 0.47 ]
 [0.145 0.594 0.148 0.873 0.02  0.966 0.689]]
Rank of right matrix: 3

Summed matrix:
[[0.618 0.168 1.226 0.205]
 [1.241 1.074 0.75  1.291]
 [0.806 1.133 0.674 0.814]
 [1.172 1.921 0.338 0.954]
 [0.605 1.548 1.007 0.358]
 [1.355 0.579 0.781 1.513]
 [1.61  1.13  0.889 1.494]]
Rank of summed matrix: 4

Product matrix:
[[0.456 0.69  0.19  0.624 0.113 0.959 0.774]
 [0.386 1.209 0.538 1.6   0.316 1.562 1.42 ]
 [0.561 1.307 0.503 1.554 0.292 1.74  1.512]
 [0.254 0.973 0.817 1.276 0.651 0.701 1.193]
 [0.717 1.565 0.871 1.74  0.648 1.656 1.837]
 [0.244 0.885 0.454 1.21  0.285 1.072 1.051]
 [0.499 1.408 0.

## 2. Transpose and Gram Identities
Based on [Lecture 67](https://wgu.udemy.com/course/linear-algebra-theory-and-implementation/learn/lecture/10500654#overview), these identities establish that the fundamental dimensionality of a dataset remains invariant under transposition and "Gram-style" self-multiplication.

### Core Identity
The numerical rank of a matrix is equal to the rank of its transpose, as well as the rank of its products with its transpose:
$$rank(A) = rank(A^T) = rank(A^TA) = rank(AA^T)$$

### Complexity Analysis
* **Transpose ($A^T$):** $O(m \cdot n)$ — Requires visiting every element to swap indices.
* **Gram Product ($A^TA$ or $AA^T$):** $O(n^3)$ (for square matrices) — Involves a standard matrix multiplication bottleneck.
* **Rank Verification:** $O(n^3)$ — NumPy's `matrix_rank` utilizes SVD, which is computationally expensive for large datasets.

### Why Gram Matrices Matter
In machine learning and statistics, $A^TA$ (the covariance matrix) is often easier to work with than the raw data matrix $A$, but these identities guarantee that we haven't lost any "latent" dimensions during the transformation.

In [3]:
#======================================================================
# SECTION: TRANSPOSE AND GRAM IDENTITIES
# COMPLEXITY: O(n^3) for Gram products (A @ A.T).
# DATA SHAPE: A (m x n) -> A.T @ A (n x n) and A @ A.T (m x m).
# MATH: rank(A) = rank(A^T) = rank(A^T @ A) = rank(A @ A.T)
# TOOLS: np.random.default_rng(), .T (transpose), np.linalg.matrix_rank()
# PROTOCOL: Think-First Whiteboard Applied
#======================================================================

MATRIX_SIZE = 5

# Use NumPy's newer random number generator API.
# Keeping it in one variable makes it easy to reuse throughout the notebook.
rng = np.random.default_rng()


def generate_symmetric_matrix(size, random_generator):
    """Generate a square symmetric matrix with random values."""
    # Start with a random square matrix.
    random_matrix = random_generator.random((size, size))

    # Add the matrix to its transpose so entry (i, j) equals entry (j, i).
    return random_matrix + random_matrix.T


def matrix_rank(matrix):
    """Return the numerical rank of a matrix."""
    # NumPy estimates rank numerically using singular values.
    # This is more reliable than trying to row-reduce floating-point matrices by hand.
    return np.linalg.matrix_rank(matrix)


def gram_matrix(matrix):
    """Return matrix @ matrix.T."""
    # A @ A.T is a Gram-style matrix.
    # It preserves rank relationships that are useful for testing:
    # rank(A) = rank(A.T) = rank(A.T @ A) = rank(A @ A.T)
    return matrix @ matrix.T


def format_rank(label, matrix):
    """Format a matrix rank line for display."""
    return f"Rank of {label}: {matrix_rank(matrix)}"


def print_matrix_rank_summary(matrix, matrix_name="A"):
    """Print rank relationships for a matrix, its transpose, and Gram products."""
    # The transpose should have the same rank as the original matrix.
    transpose_matrix = matrix.T

    # These products are square matrices built from A.
    # For a given matrix A, A.T @ A and A @ A.T have the same rank as A.
    transpose_product = transpose_matrix @ matrix
    product_with_transpose = gram_matrix(matrix)

    print(format_rank(matrix_name, matrix))
    print(format_rank(f"{matrix_name}^T{matrix_name}", transpose_product))
    print(format_rank(f"{matrix_name}{matrix_name}^T", product_with_transpose))
    print(format_rank(f"{matrix_name}^T", transpose_matrix))


# Create two symmetric matrices to compare rank behavior under addition and multiplication.
first_matrix = generate_symmetric_matrix(MATRIX_SIZE, rng)
second_matrix = generate_symmetric_matrix(MATRIX_SIZE, rng)

# First, verify the standard rank relationships for A, A.T, A.T @ A, and A @ A.T.
print_matrix_rank_summary(first_matrix)

# Build Gram-style matrices from both symmetric matrices.
first_gram_matrix = gram_matrix(first_matrix)
second_gram_matrix = gram_matrix(second_matrix)

# Addition can combine column/row spaces, but rank is still bounded.
gram_matrix_sum = first_gram_matrix + second_gram_matrix

# Multiplication cannot exceed the rank of the lowest-rank factor.
gram_matrix_product = first_gram_matrix @ second_gram_matrix

print(format_rank("the sum of AA^T + BB^T", gram_matrix_sum))
print(format_rank("the product of AA^T and BB^T", gram_matrix_product))

Rank of A: 5
Rank of A^TA: 5
Rank of AA^T: 5
Rank of A^T: 5
Rank of the sum of AA^T + BB^T: 5
Rank of the product of AA^T and BB^T: 5
